# Update 1 · Evolución del notebook 04

**Primera parte conservada:** 31 celdas del notebook original.

**Para el ejercicio de grandes datos y GPU, empieza directamente en «Laboratorio P04 · Analizar tres periodos PQRS con pandas y Polars GPU», en L1.**

Ese laboratorio contiene la descarga, la conversión y los ejercicios completos. No requiere ejecutar P02 ni la primera parte. Se parte del kernel GPU ya preparado.


# P04 e I01. Rendimiento y traza finita

P04: clase 7, 23 de octubre; I01: clase 8, 24 de octubre de 2026.
Preparado por el PhD Esteban Hernández, CyberColombia.

[Enunciado P04](../../talleres/P04.md) · [I01](../../talleres/I01.md) · [WSL](../../docs/instalacion-wsl.md) · [Linux/macOS](../../docs/instalacion-linux-macos.md)

El viernes el profesor demuestra; el sábado los estudiantes ejecutan y entregan en clase. Ejecutar las celdas en orden con el kernel del único `.venv`. Cada lectura, consulta y escritura queda visible. No se importan scripts del curso.

## 1. Preparar y localizar los datos
Desde la terminal en `kit/`, con `.venv` activo:
```bash
python 00_datos.py --descargar pqrs nasa
python 00_datos.py --verificar pqrs nasa
```
Primero ejecutar [02 Parquet y contratos](02_Parquet_y_contratos.ipynb): allí se leen los CSV, se convierten los tipos y se escriben el CSV proyectado, el Parquet y sus particiones. Aquí se leen esos productos; no se regeneran silenciosamente. Las muestras contienen 3.000 reportes; no permiten concluir sobre rendimiento nacional.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Editar esta ruta para que coincida con el clon local.
# Linux/macOS: Path.home() / "bigdata/big-data-postgraduate"
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas" / "pandas/P04"
assert KIT.is_dir(), "Corrige repositorio antes de continuar"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
entrada = KIT / "salidas/pandas/P02"
archivo_parquet = entrada / "pqrs.parquet"
archivo_csv = entrada / "pqrs_proyeccion.csv"
particiones = entrada / "pqrs_particionado"
assert archivo_parquet.is_file(), "Ejecuta el notebook 02"
assert archivo_csv.is_file() and list(particiones.glob("**/*.parquet"))
control_entrada = json.loads((entrada / "control_formatos_pandas.json").read_text())
assert control_entrada["filas"] == 3000, "Este bloque mide las muestras de P02"
columnas = ["corte_archivo", "anio", "mes_num", "afec_cod_depto"]
# Importar antes de medir: excluimos el coste de importación de las bibliotecas.
import polars as pl
import duckdb
from time import perf_counter
import platform
mediciones = []
display(pd.DataFrame([{"python": platform.python_version(), "pandas": pd.__version__,
                       "polars": pl.__version__, "duckdb": duckdb.__version__,
                       "sistema": platform.platform()}]))


## 2. Comparar la misma consulta en cinco variantes
Agrupar **todos los años** por corte, año, mes y departamento, incluyendo claves nulas, y contar filas. Cada celda mide lectura + agregación + obtención del resultado en pandas. `%%time` informa CPU y **Wall time**; `perf_counter` guarda el tiempo transcurrido del bloque para el informe. Polars incluye `to_pandas`; DuckDB incluye la apertura/cierre de conexión.

La ordenación, normalización de tipos, comprobación y visualización se realizan después y quedan fuera del intervalo registrado. Ejecutar una vuelta de calentamiento, vaciar `mediciones` con la celda indicada y realizar tres vueltas, alternando el orden de los motores. No volver a ejecutar la configuración entre vueltas. La caché del sistema puede persistir; no llamar a esto una prueba con caché fría. Este ejercicio comparte el proceso del kernel y no mide RSS ni arranque de procesos.


### pandas: lectura inmediata y groupby


In [ ]:
%%time
inicio = perf_counter()
datos_pandas = pd.read_parquet(archivo_parquet, columns=columnas)
resultado_pandas = (datos_pandas.groupby(columnas, dropna=False)
    .size().reset_index(name="reportes"))
segundos = perf_counter() - inicio
mediciones.append({"motor": "pandas", "segundos": segundos})


### Polars: lectura diferida y ejecución explícita
`scan_parquet` prepara el plan; `collect` lo ejecuta. Medir solo la creación de `consulta` no mide el procesamiento.


In [ ]:
%%time
inicio = perf_counter()
consulta = (pl.scan_parquet(archivo_parquet)
    .group_by(columnas).agg(pl.len().alias("reportes")))
resultado_polars = consulta.collect(engine="streaming").to_pandas()
segundos = perf_counter() - inicio
mediciones.append({"motor": "polars", "segundos": segundos})


### DuckDB: SQL sobre Parquet
Las tres variantes siguientes usan exactamente el mismo SQL. El CSV se lee como texto para conservar los ceros de los códigos; por ello incluye un coste distinto de interpretación del formato.


In [ ]:
sql = """
SELECT corte_archivo, anio, mes_num, afec_cod_depto, COUNT(*) AS reportes
FROM reportes
GROUP BY corte_archivo, anio, mes_num, afec_cod_depto
"""


### duckdb


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(archivo_parquet)).create_view("reportes")
    resultado_duckdb = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb", "segundos": segundos})


### duckdb_particionado


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(particiones / "**/*.parquet"), hive_partitioning=True).create_view("reportes")
    resultado_duckdb_particionado = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_particionado", "segundos": segundos})


### duckdb_csv


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_csv(str(archivo_csv), header=True, all_varchar=True).create_view("reportes")
    resultado_duckdb_csv = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_csv", "segundos": segundos})


## 3. Verificar antes de comparar tiempos
El CSV devuelve claves numéricas como texto; armonizamos tipos fuera de la medición. No basta con comparar el total: cada grupo debe coincidir. Inspeccionar primero los tipos originales. Repetir esta comprobación después de cada vuelta.


In [ ]:
resultados = {"pandas": resultado_pandas, "polars": resultado_polars,
              "duckdb": resultado_duckdb, "duckdb_particionado": resultado_duckdb_particionado,
              "duckdb_csv": resultado_duckdb_csv}
comparables = {}
for motor, resultado in resultados.items():
    print(motor, resultado.dtypes.to_dict())
    normalizado = resultado.copy()
    for columna in ["anio", "mes_num", "reportes"]:
        normalizado[columna] = pd.to_numeric(normalizado[columna], errors="raise").astype("Int64")
    normalizado["corte_archivo"] = normalizado["corte_archivo"].astype("string")
    normalizado["afec_cod_depto"] = normalizado["afec_cod_depto"].astype("string").str.zfill(2)
    comparables[motor] = normalizado.sort_values(columnas).reset_index(drop=True)
    assert normalizado["reportes"].sum() == 3000
for motor, resultado in comparables.items():
    pd.testing.assert_frame_equal(comparables["pandas"], resultado)
display(comparables["pandas"])
print("Los cinco resultados coinciden grupo por grupo")


### Excluir el calentamiento
Ejecutar la siguiente celda **una vez**, tras verificar la primera vuelta. Después repetir únicamente las cinco celdas medidas y la comprobación anterior tres veces; alternar su orden. No ejecutar de nuevo esta celda al finalizar.


In [ ]:
mediciones = []


### Guardar las tres repeticiones
Ejecutar tras completar las tres vueltas. Este archivo documenta el alcance del kernel; no sustituye las mediciones de RSS en procesos independientes.


In [ ]:
tiempos = pd.DataFrame(mediciones)
assert len(tiempos) == 15, "Faltan las tres vueltas de cinco motores después del calentamiento"
assert tiempos.groupby("motor").size().eq(3).all()
tiempos["repeticion"] = tiempos.groupby("motor").cumcount() + 1
tiempos["alcance"] = "kernel: lectura, agregación y resultado pandas; 3000 filas"
resumen = tiempos.groupby("motor")["segundos"].agg(["median", "min", "max"]).reset_index()
tiempos.to_csv(OUT / "benchmark_notebook.csv", index=False)
resumen.to_csv(OUT / "benchmark_notebook_resumen.csv", index=False)
comparables["pandas"].to_csv(OUT / "conteos_verificados.csv", index=False)
display(resumen)


## 4. Escala real y automatización, después de comprender las consultas
[P04](../../talleres/P04.md) conserva la extensión de cinco variantes en procesos independientes, dos hilos, calentamiento, tres repeticiones y sondeo RSS cada 10 ms. Revisar `trabajador` y `benchmark` en `kit/pqrs_talleres.py`: identificar en el código cada consulta que acabas de ejecutar antes de usar la automatización.

Los tiempos del CLI incluyen arranque y exportación; **no mezclar** esas mediciones con las del kernel. El CLI escribe `benchmark_pqrs.csv` y su resumen; el notebook usa otros nombres para distinguir el alcance. Registrar máquina, versiones e hilos. Las bibliotecas del kernel pueden usar distintos números de hilos: esta comparación describe la configuración local, no aísla el efecto del motor.

El volumen completo se descarga explícitamente según P04; reservar al menos 8 GB de disco más el entorno. Con 8 GB de RAM mantener pandas en muestras. Los completos concilian 2.444.766 reportes. No cambiar la ruta de este bloque de muestras a los completos sin revisar su preparación y memoria.

**Detenerse aquí en la clase 7.** Retomar eventos en la clase 8.

## 5. Leer las observaciones NASA
`json.load` lee el documento; `DataFrame` convierte el diccionario de parámetros en filas. Convertimos las fechas y separamos los faltantes (-999), sin inventar lluvia. La fecha diaria se usa como tiempo de evento por convención didáctica.


In [ ]:
with open(RAW / "nasa.json", encoding="utf-8") as archivo:
    nasa = json.load(archivo)
lluvia = pd.Series(nasa["properties"]["parameter"]["PRECTOTCORR"], name="lluvia_mm")
observaciones = lluvia.rename_axis("event_id").reset_index()
observaciones["fecha_evento"] = pd.to_datetime(observaciones["event_id"], format="%Y%m%d")
observaciones["lluvia_mm"] = pd.to_numeric(observaciones["lluvia_mm"], errors="raise").replace(-999, float("nan"))
assert observaciones["event_id"].is_unique
display(observaciones.head(12))


## 6. Construir el orden de entrega y unir los valores reales
Primero llegan los días 1–4 y 6–12; luego se reintenta el 6 y finalmente llega el 5. `merge(validate="many_to_one")` permite entregas repetidas pero exige una observación por fecha. Inspeccionar la tabla antes de aplicar reglas.


In [ ]:
dias = [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 6, 5]
entregas = pd.DataFrame({"event_id": [f"202501{dia:02d}" for dia in dias]})
entregas["orden_entrega"] = range(1, len(entregas) + 1)
entregas = entregas.merge(observaciones, on="event_id", how="left", validate="many_to_one", indicator=True)
assert entregas["_merge"].eq("both").all() and entregas["lluvia_mm"].notna().all()
display(entregas)


## 7. Recorrer la traza paso a paso
Antes de cada entrega, el límite es la fecha máxima observada menos tres días. Primero comprobamos si el identificador ya fue aceptado; después, si la fecha es anterior al límite. Una fecha igual al límite se admite. La lista `traza` conserva la decisión y el límite usado; `vistos` solo guarda aceptados. Esta regla finita no reproduce toda la semántica de Spark.


In [ ]:
vistos = set()
ultima_fecha = None
traza = []
for entrega in entregas.itertuples(index=False):
    limite = ultima_fecha - pd.Timedelta(days=3) if ultima_fecha is not None else None
    if entrega.event_id in vistos:
        estado = "duplicado"
    elif limite is not None and entrega.fecha_evento < limite:
        estado = "tardio"
    else:
        estado = "aceptado"
        vistos.add(entrega.event_id)
    traza.append({"orden_entrega": entrega.orden_entrega, "event_id": entrega.event_id,
                  "fecha_evento": entrega.fecha_evento, "lluvia_mm": entrega.lluvia_mm,
                  "watermark_antes": limite, "estado": estado})
    ultima_fecha = max(ultima_fecha, entrega.fecha_evento) if ultima_fecha is not None else entrega.fecha_evento
    print(entrega.orden_entrega, entrega.event_id, "límite:", limite, estado)
traza = pd.DataFrame(traza)


## 8. Conciliar y escribir las evidencias
Comparar entregas, fechas distintas y aceptados. Explicar qué observación falta en la suma aceptada y proponer una corrección antes de continuar con Spark en E11.


In [ ]:
aceptados = traza.loc[traza["estado"].eq("aceptado")]
control = {"entregas": len(traza), "registros_reales_distintos": int(traza["event_id"].nunique()),
           "aceptados": len(aceptados), "duplicados": int(traza["estado"].eq("duplicado").sum()),
           "tardios": int(traza["estado"].eq("tardio").sum()),
           "suma_aceptada_mm": float(aceptados["lluvia_mm"].sum()),
           "suma_fechas_unicas_mm": float(entregas.drop_duplicates("event_id")["lluvia_mm"].sum())}
assert (control["entregas"], control["registros_reales_distintos"], control["aceptados"], control["duplicados"], control["tardios"]) == (13, 12, 11, 1, 1)
assert abs(control["suma_aceptada_mm"] - 41.44) < 1e-8
traza.to_csv(OUT / "traza_eventos_nasa.csv", index=False)
with open(OUT / "control_eventos_intro.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
recuperada = pd.read_csv(OUT / "traza_eventos_nasa.csv", dtype={"event_id": "string"},
                         parse_dates=["fecha_evento", "watermark_antes"])
pd.testing.assert_frame_equal(traza.astype({"event_id": "string"}), recuperada)
display(control)
display(recuperada)


## Entrega
Adjuntar tiempos, resultados equivalentes, configuración y alcance de la medición; distinguir notebook y CLI. Entregar la traza y explicar el reintento y el atraso. Modificar una condición de consulta, anticipar el resultado y verificarlo en todos los motores. No sumar lluvia con PQRS ni fabricar tiempos de evento para reportes.


# Laboratorio P04 · Analizar tres periodos PQRS con pandas y Polars GPU

La GPU ya está verificada y el kernel GPU está seleccionado. Ahora vamos a usarla.

**Qué haremos:** descargar tres CSV completos, convertirlos aquí a Parquet,
contar los reportes con pandas y ejecutar las mismas preguntas con Polars GPU.
Al final compararemos resultados y tiempos. Polars CPU será una referencia adicional.

Usaremos **2023-II, 2024-I y 2024-II**: las fuentes de referencia registran un total de
**2.444.766 reportes** y aproximadamente **1,82 GB de CSV**. No son muestras.

**Empieza aquí y ejecuta las celdas en orden, de L1 a L11. No necesitas ejecutar
la primera parte de este notebook ni preparar archivos en otro notebook.**

1. Indicar dónde está el repositorio.
2. Comprobar los CSV y descargar los que falten.
3. Mirar unas filas para reconocer los datos y sus separadores.
4. Convertir todos los periodos a Parquet y comprobar sus filas.
5. Formular las preguntas de análisis.
6. Resolverlas con pandas.
7. Preparar y ejecutar las consultas Polars en CPU.
8. Ejecutarlas en la GPU y comprobar que los resultados coinciden.
9. Repetir las mediciones tres veces.
10. Comparar tiempos y guardar las evidencias.
11. Explicar lo observado.


## L1. Indicar la carpeta del repositorio

**Haz esto:** ejecuta la celda siguiente. Si no encuentra el repositorio,
cambia `REPOSITORIO = None` por la ruta de tu clon y vuelve a ejecutarla.
En WSL, una ruta de Windows empieza por `/mnt/c/`, no por `C:\`.

Deja los tres periodos indicados. La celda crea estas carpetas si no existen:

| Carpeta | Qué contendrá |
|---|---|
| `kit/data/pqrs_completos/` | Los tres CSV descargados. |
| `kit/salidas/pandas/P04_update1/parquet/` | Un Parquet y su control por periodo. |
| `kit/salidas/pandas/P04_update1/` | Resultados del análisis y tiempos. |

**Comprueba:** deben aparecer tres periodos y el total esperado. Reserva espacio
para los CSV, los Parquet y los temporales. `BLOQUE = 50_000` indica cuántas
filas procesará pandas en cada lote; no limita toda la memoria del proceso.


In [ ]:
from pathlib import Path
from time import perf_counter
from importlib import metadata
from IPython.display import display
import csv, hashlib, json, os, platform, shutil, subprocess, sys
import urllib.request
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

REPOSITORIO = None  # Ejemplo WSL: Path('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate')
PERIODOS = ['2023_II', '2024_I', '2024_II']
BLOQUE = 50_000
REPETICIONES = 3
DISPOSITIVO_CUDA = 0  # Índice CUDA, no el número de GPU del Administrador de tareas
if REPOSITORIO is None:
    REPOSITORIO = next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'kit/pqrs_fuentes.json').is_file()), None)
assert REPOSITORIO is not None, 'Configurar REPOSITORIO'
REPOSITORIO = Path(REPOSITORIO).expanduser().resolve()
KIT = REPOSITORIO / 'kit'
CSV_DIR = KIT / 'data/pqrs_completos'
OUT = KIT / 'salidas/pandas/P04_update1'
PARQUET_DIR = OUT / 'parquet'
for carpeta in [CSV_DIR, OUT, PARQUET_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)
COLUMNAS_CSV = ['OBJECTID',
 'periodo',
 'mes',
 'pqr_canal',
 'pet_cod_depto',
 'pet_cod_mpio',
 'id_afec',
 'afec_parentesco',
 'afec_genero',
 'afec_edad',
 'afec_edadr',
 'afec_educ',
 'afec_regafiliacion',
 'afec_getnico',
 'afec_pobespecial',
 'afec_cod_depto',
 'afec_cod_mpio',
 'ent_nombre',
 'ent_tipovig_sns',
 'ent_cod_sns',
 'ent_alias_sn',
 'ent_cod_depto',
 'ent_cod_mpio',
 'cod_macromot',
 'macromotivo',
 'cod_motgen',
 'motivo_general',
 'cod_motesp',
 'motivo_especifico',
 'cod_tipo_motesp',
 'tipo_de_motivo_especifico',
 'cod_subtipo_motesp',
 'subtipo_de_motivo_especifico',
 'patologia_1',
 'patologia_tipo',
 'cie_10',
 'alto_costo',
 'clasificacion_de_riesgo']

FUENTES = [{'periodo': '2023_II',
  'archivo': 'PQRD_2023_II.csv',
  'url': 'https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/515d4c6366854f549b24c704320f10f4/data',
  'filas': 716697,
  'bytes': 532695991,
  'separador': ';',
  'sha256': '78c21edacfc5b60afd1b8e27cffcca41ee1c1faf65d33a410a4d792739e8aec8'},
 {'periodo': '2024_I',
  'archivo': 'PQRD_2024_I.csv',
  'url': 'https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b52330a16b3940c39d38cb164c9dc014/data',
  'filas': 781601,
  'bytes': 576710688,
  'separador': ',',
  'sha256': 'f090c0a1eee399bfa6512fbaf9451d422ac3fe0fd0789a7819e0d8df0cd5e956'},
 {'periodo': '2024_II',
  'archivo': 'PQRD_2024_II.csv',
  'url': 'https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/b257c9907b754b0cabb55a32e706bdee/data',
  'filas': 946468,
  'bytes': 706836370,
  'separador': ';',
  'sha256': '03ec2dd5363673190239085c49b1e731d77ead8e0356817a2b34c44765f99f3d'}]
for fuente in FUENTES:
    fuente['columnas'] = COLUMNAS_CSV
    fuente['codificacion'] = 'utf-8-sig'
assert len(FUENTES) == 3
assert [f['periodo'] for f in FUENTES] == PERIODOS
TOTAL_ESPERADO = sum(f['filas'] for f in FUENTES)
assert BLOQUE > 0 and REPETICIONES >= 3
CLAVES = ['corte_archivo', 'anio', 'mes_num', 'afec_cod_depto']
CONSULTAS = {'territorio_mes': CLAVES,
             'territorio_mes_entidad': CLAVES + ['ent_nombre']}
print('Intérprete:', sys.executable)
print('Versiones:', {'pandas': pd.__version__, 'polars': pl.__version__, 'pyarrow': pa.__version__})
display(pd.DataFrame([{k: f[k] for k in ['periodo', 'archivo', 'filas', 'bytes', 'separador', 'url']}
                      for f in FUENTES]))
print('Total esperado:', TOTAL_ESPERADO, 'reportes')
print('Libres en disco (GiB):', shutil.disk_usage(KIT).free / 2**30)


## L2. Descargar y comprobar los tres CSV completos

Usaremos el descargador del curso. **En una terminal Ubuntu/WSL, desde la raíz
del repositorio**, entra en `kit` y ejecuta:

```bash
cd kit
python 00_datos.py --descargar pqrs-completos
python 00_datos.py --verificar pqrs-completos
```

El primer comando descarga secuencialmente **2023-II, 2024-I y 2024-II** en
`kit/data/pqrs_completos/` (aproximadamente **1,82 GB**). Si un CSV ya existe y
su huella coincide, lo conserva. Si difiere, informa el problema sin sobrescribirlo.
El segundo comando comprueba huellas, cabeceras, filas y estructura.

**Deben quedar estos archivos:**

| Periodo | Archivo | Reportes esperados |
|---|---|---:|
| 2023-II | `PQRD_2023_II.csv` | 716.697 |
| 2024-I | `PQRD_2024_I.csv` | 781.601 |
| 2024-II | `PQRD_2024_II.csv` | 946.468 |

**También puedes ejecutar directamente la celda siguiente:** invoca los mismos
comandos con el Python del kernel, sin cambiar de entorno. Si ya los ejecutaste
en terminal, conservará los archivos correctos y volverá a verificarlos.
No hace falta pegar las URL ni usar otro notebook.

**Comprueba:** deben aparecer los tres archivos con estado `coincide` y sumar
**2.444.766 reportes**. Si un comando falla, la celda se detiene. Resuelve el
problema antes de pasar a L3. Estos comandos descargan y verifican CSV;
la conversión a Parquet la harás en L4.


In [ ]:
# Ejecutar desde Jupyter los mismos comandos explicados arriba.
# sys.executable conserva el Python y el entorno del kernel actual.
subprocess.run([sys.executable, '00_datos.py', '--descargar', 'pqrs-completos'],
               cwd=KIT, check=True)
subprocess.run([sys.executable, '00_datos.py', '--verificar', 'pqrs-completos'],
               cwd=KIT, check=True)

descargas = json.loads((KIT / 'salidas/verificacion.json').read_text(encoding='utf-8'))
assert len(descargas) == 3
assert {r['archivo'] for r in descargas} == {f['archivo'] for f in FUENTES}
assert all(r['estado'] == 'coincide' for r in descargas)
assert sum(r['filas'] for r in descargas) == TOTAL_ESPERADO
print('Tres CSV completos descargados y verificados:', TOTAL_ESPERADO, 'reportes')
display(pd.DataFrame(descargas)[['archivo', 'estado', 'filas', 'bytes']])

# En L4 también comprobaremos la huella de cada Parquet sin cargarlo en RAM.
def sha256_archivo(archivo):
    huella = hashlib.sha256()
    with archivo.open('rb') as stream:
        for bloque in iter(lambda: stream.read(1024 * 1024), b''):
            huella.update(bloque)
    return huella.hexdigest()


## L3. Reconocer las columnas y los separadores

**Haz esto:** ejecuta la celda para ver cinco filas de cada periodo.

Observa `OBJECTID`, `periodo`, `mes`, `afec_cod_depto` y `ent_nombre`.
Cada archivo tiene su propio separador; por eso no usamos uno único para los tres.
Los códigos se leen como texto para conservar ceros iniciales.

**Responde:** ¿qué diferencia hay entre el corte `2023_II` y el valor de la
columna `periodo`? El primero identifica el archivo semestral; la columna
de la fuente informa el año. El mes se toma de la columna `mes`.


In [ ]:
for fuente in FUENTES:
    vista = pd.read_csv(CSV_DIR / fuente['archivo'], sep=fuente['separador'],
        encoding=fuente.get('codificacion', 'utf-8-sig'), dtype=str,
        keep_default_na=False, nrows=5)
    print('Periodo:', fuente['periodo'], 'Columnas:', len(vista.columns))
    display(vista[['OBJECTID', 'periodo', 'mes', 'afec_cod_depto', 'ent_nombre']])


## L4. Convertir los tres CSV a Parquet

**Haz esto:** ejecuta esta celda. La conversión está desarrollada aquí:

1. Comprueba si cada periodo ya tiene un Parquet válido y su archivo de control.
2. Si no lo tiene, lee el CSV en bloques de 50.000 filas usando pandas.
3. Conserva las 38 columnas originales como texto, incluidos vacíos y duplicados.
4. Añade `corte_archivo`, `anio` y `mes_num`. Una conversión numérica inválida
   queda nula; no se elimina el reporte.
5. Convierte cada bloque a una tabla Arrow y lo escribe en Parquet con Zstandard.
6. Comprueba el total de filas del periodo antes de publicar el archivo terminado.

Al reejecutar, solo reutiliza un Parquet si coinciden fuente, esquema, filas y
huella. Si falta o está incompleto, lo vuelve a generar.

**Comprueba:** la tabla final debe tener **tres periodos**, cada uno con su
conteo correcto. La suma debe ser **2.444.766 reportes**. Solo entonces seguimos.
Ambos motores analizarán estos mismos Parquet, sin preparar productos diferentes
para favorecer a uno de ellos. Este tiempo es conversión, no tiempo de consulta.


In [ ]:
conversiones = []
for fuente in FUENTES:
    periodo = fuente['periodo']
    parquet = PARQUET_DIR / f'{periodo}.parquet'
    control_path = PARQUET_DIR / f'{periodo}.control.json'
    esquema = pa.schema([(nombre, pa.string()) for nombre in fuente['columnas']] +
        [('corte_archivo', pa.string()), ('anio', pa.int64()), ('mes_num', pa.int64())])
    valido = False
    if parquet.is_file() and control_path.is_file():
        try:
            anterior = json.loads(control_path.read_text())
            pf = pq.ParquetFile(parquet)
            valido = (anterior['sha256_csv'] == fuente['sha256'] and
                anterior['filas'] == fuente['filas'] == pf.metadata.num_rows and
                pf.schema_arrow.equals(esquema, check_metadata=False) and
                anterior['sha256_parquet'] == sha256_archivo(parquet))
        except (OSError, ValueError, KeyError, pa.ArrowException):
            valido = False
    if valido:
        conversiones.append({**anterior, 'estado': 'reutilizado_verificado'})
        print(periodo, 'Parquet ya convertido y verificado', flush=True)
        continue
    print('Convirtiendo CSV → Parquet:', periodo, flush=True)
    inicio = perf_counter()
    temporal = parquet.with_suffix('.parquet.part')
    filas = 0
    with pq.ParquetWriter(temporal, esquema, compression='zstd') as escritor:
        lector = pd.read_csv(CSV_DIR / fuente['archivo'], sep=fuente['separador'],
            encoding=fuente.get('codificacion', 'utf-8-sig'), dtype=str,
            keep_default_na=False, chunksize=BLOQUE)
        for bloque in lector:
            assert list(bloque.columns) == fuente['columnas']
            bloque['corte_archivo'] = periodo
            bloque['anio'] = pd.to_numeric(bloque['periodo'], errors='coerce').astype('Int64')
            bloque['mes_num'] = pd.to_numeric(bloque['mes'], errors='coerce').astype('Int64')
            tabla = pa.Table.from_pandas(bloque, schema=esquema, preserve_index=False, safe=True)
            escritor.write_table(tabla)
            filas += len(bloque)
    segundos_conversion = perf_counter() - inicio
    assert filas == fuente['filas'], f'Conteo distinto en {periodo}: {filas}'
    assert pq.ParquetFile(temporal).metadata.num_rows == filas
    temporal.replace(parquet)
    control = {'periodo': periodo, 'archivo': str(parquet), 'filas': filas,
        'columnas': len(esquema), 'sha256_csv': fuente['sha256'],
        'sha256_parquet': sha256_archivo(parquet),
        'bytes_csv': fuente['bytes'], 'bytes_parquet': parquet.stat().st_size,
        'segundos_conversion': segundos_conversion, 'bloque_pandas': BLOQUE}
    control_path.write_text(json.dumps(control, ensure_ascii=False, indent=2))
    conversiones.append({**control, 'estado': 'convertido_verificado'})

assert len(conversiones) == len(FUENTES), 'No continuar: faltan periodos convertidos'
assert sum(c['filas'] for c in conversiones) == TOTAL_ESPERADO
ARCHIVOS_PARQUET = [PARQUET_DIR / f"{f['periodo']}.parquet" for f in FUENTES]
assert all(p.is_file() for p in ARCHIVOS_PARQUET)
display(pd.DataFrame(conversiones)[['periodo', 'estado', 'filas', 'columnas',
    'bytes_csv', 'bytes_parquet', 'segundos_conversion']])
print('TODOS los periodos descargados, convertidos y verificados. Se puede analizar.')


## L5. Definir las dos preguntas

**Ejercicio 1 — Volumen territorial:** contar reportes por corte, año, mes y
departamento. Una fila del resultado representa una combinación de esas claves.

**Ejercicio 2 — Entidad y territorio:** repetir el conteo añadiendo `ent_nombre`.
Antes de ejecutarlo, anticipa si habrá más grupos y más memoria de trabajo.

Contamos reportes, no personas únicas. Conservamos claves vacías y nulas.
No eliminamos duplicados durante este ejercicio.

**Qué mediremos:** lectura del mismo Parquet + agrupación + resultado final en
memoria CPU. Para Polars GPU esto incluye devolver el resultado al host.
Descarga, conversión, comprobación y escritura del informe se miden por separado
o quedan fuera del benchmark. pandas trabajará por lotes; Polars CPU usará
streaming y Polars GPU utilizará cuDF.


## L6. Resolver los ejercicios con pandas

**Haz esto:** ejecuta la celda y observa los resultados de las dos preguntas.

El recorrido es: leer columnas de un lote Parquet → contar por las claves →
guardar conteos parciales → sumar parciales de todos los lotes y periodos.
Por ejemplo, si una clave aparece 20 veces en un lote y 30 en otro, el resultado
es 50 reportes. No necesitamos cargar las 38 columnas completas para contarlos.

**Comprueba:** en cada consulta la suma de `reportes` debe ser **2.444.766**.
Anota cuántos grupos produce cada consulta. Estos resultados serán la referencia
para comprobar Polars. El primer tiempo se guarda aparte de las repeticiones.

Los conteos parciales también consumen RAM; trabajar por lotes no garantiza
que cualquier consulta quepa en memoria.


In [ ]:
def analizar_pandas(claves):
    parciales = []
    for archivo in ARCHIVOS_PARQUET:
        for lote in pq.ParquetFile(archivo).iter_batches(batch_size=BLOQUE, columns=claves):
            datos = lote.to_pandas()
            for nombre in ['anio', 'mes_num']:
                datos[nombre] = pd.to_numeric(datos[nombre], errors='raise').astype('Int64')
            parcial = datos.groupby(claves, dropna=False).size().reset_index(name='reportes')
            parciales.append(parcial)
    acumulados = pd.concat(parciales, ignore_index=True)
    return acumulados.groupby(claves, dropna=False, as_index=False)['reportes'].sum()

referencias = {}
primeras = []
for nombre, claves in CONSULTAS.items():
    inicio = perf_counter()
    resultado = analizar_pandas(claves)
    primeras.append({'consulta': nombre, 'motor': 'pandas_cpu',
                     'segundos': perf_counter() - inicio})
    assert int(resultado['reportes'].sum()) == TOTAL_ESPERADO
    referencias[nombre] = resultado
    print(nombre, 'reportes:', resultado['reportes'].sum(), 'grupos:', len(resultado))
    display(resultado.head(10))


## L7. Escribir las mismas consultas con Polars

**Haz esto:** lee las dos consultas de la celda y ejecútala.

`scan_parquet` describe la lectura, `group_by` define las claves y `pl.len()`
cuenta filas. `collect` ejecuta el trabajo. Primero usamos CPU para tener una
referencia adicional; en L8 ejecutaremos esos mismos planes en GPU.

**Comprueba:** deben coincidir todos los grupos con pandas, no solo el total.
La función `comparable` ordena y armoniza tipos para hacer esa comprobación
fuera del tiempo medido.


In [ ]:
# Ejercicio 1: contar reportes por territorio y mes.
consulta_territorio = (
    pl.scan_parquet(ARCHIVOS_PARQUET)
    .select(CLAVES)
    .group_by(CLAVES)
    .agg(pl.len().alias('reportes'))
)

# Ejercicio 2: añadir entidad a las claves de agrupación.
claves_entidad = CLAVES + ['ent_nombre']
consulta_entidad = (
    pl.scan_parquet(ARCHIVOS_PARQUET)
    .select(claves_entidad)
    .group_by(claves_entidad)
    .agg(pl.len().alias('reportes'))
)

consultas_polars = {'territorio_mes': consulta_territorio,
                   'territorio_mes_entidad': consulta_entidad}
print(consulta_territorio.explain())

def comparable(resultado, claves):
    copia = resultado.copy()
    for nombre in ['anio', 'mes_num', 'reportes']:
        copia[nombre] = pd.to_numeric(copia[nombre], errors='raise').astype('Int64')
    for nombre in set(claves) - {'anio', 'mes_num'}:
        copia[nombre] = copia[nombre].astype('string')
    return copia.sort_values(claves, na_position='last').reset_index(drop=True)[claves + ['reportes']]

for nombre, claves in CONSULTAS.items():
    inicio = perf_counter()
    resultado = consultas_polars[nombre].collect(engine='streaming').to_pandas()
    primeras.append({'consulta': nombre, 'motor': 'polars_cpu',
                     'segundos': perf_counter() - inicio})
    pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(resultado, claves))
    print(nombre, 'pandas y Polars CPU coinciden grupo por grupo')


## L8. Ejecutar los dos ejercicios en la GPU ya verificada

**Haz esto:** ejecuta esta celda con el kernel GPU que ya preparaste.
No repetimos aquí la instalación ni la detección de tarjetas.

`DISPOSITIVO_CUDA = 0` selecciona la primera NVIDIA visible para CUDA.
`raise_on_fail=True` hace que una operación no soportada produzca un error,
en lugar de ejecutar silenciosamente la consulta en CPU.

En cada ejercicio: ejecutar `collect` con el motor GPU → devolver el resultado
al host → comprobar todos los grupos contra pandas → mostrar las primeras filas.

**Comprueba:** deben aparecer dos mensajes de coincidencia. Si una consulta
falla, resuélvela antes de medir; no presentes su ejecución CPU como resultado GPU.
El tiempo mostrado corresponde a la primera ejecución; lo compararemos después
con las medianas de tres repeticiones.


In [ ]:
# La GPU ya se verificó antes del laboratorio.
engine_gpu = pl.GPUEngine(device=DISPOSITIVO_CUDA, raise_on_fail=True)
gpu_verificada = False

# Ejercicio 1 en GPU.
inicio = perf_counter()
territorio_gpu = consulta_territorio.collect(engine=engine_gpu).to_pandas()
segundos = perf_counter() - inicio
primeras.append({'consulta': 'territorio_mes', 'motor': 'polars_gpu', 'segundos': segundos})
pd.testing.assert_frame_equal(
    comparable(referencias['territorio_mes'], CLAVES),
    comparable(territorio_gpu, CLAVES))
print('Ejercicio 1: GPU y pandas coinciden. Primera ejecución:', segundos, 's')
display(territorio_gpu.head(10))

# Ejercicio 2 en GPU.
inicio = perf_counter()
entidad_gpu = consulta_entidad.collect(engine=engine_gpu).to_pandas()
segundos = perf_counter() - inicio
primeras.append({'consulta': 'territorio_mes_entidad', 'motor': 'polars_gpu', 'segundos': segundos})
pd.testing.assert_frame_equal(
    comparable(referencias['territorio_mes_entidad'], claves_entidad),
    comparable(entidad_gpu, claves_entidad))
print('Ejercicio 2: GPU y pandas coinciden. Primera ejecución:', segundos, 's')
display(entidad_gpu.head(10))
gpu_verificada = True


## L9. Repetir y comparar el rendimiento

**Haz esto:** ejecuta la celda después de completar L8. Hará un calentamiento
adicional y **tres repeticiones por consulta y motor**, alternando el orden.
Con dos consultas y tres motores, al final deben existir **18 mediciones**.

En cada vuelta se cronometra la lectura Parquet, la agrupación y el resultado
en pandas. Después se detiene el reloj y se comprueba equivalencia. La descarga,
conversión, ordenación de validación y exportación no entran en ese tiempo.

**Observa:** ¿gana siempre el mismo motor? La GPU puede ser más lenta si el
trabajo es pequeño o dominan lectura y transferencias. No borramos cachés del
sistema: las repeticiones no representan lecturas de disco frío. Registramos
hilos y configuración; este ejercicio no mide RAM pico ni VRAM pico.


In [ ]:
assert gpu_verificada, 'Primero ejecutar correctamente L8'
def ejecutar(motor, nombre):
    if motor == 'pandas_cpu':
        return analizar_pandas(CONSULTAS[nombre])
    engine = 'streaming' if motor == 'polars_cpu' else engine_gpu
    return consultas_polars[nombre].collect(engine=engine).to_pandas()

motores = ['pandas_cpu', 'polars_cpu', 'polars_gpu']
for nombre, claves in CONSULTAS.items():
    for motor in motores:
        calentamiento = ejecutar(motor, nombre)
        pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(calentamiento, claves))
        del calentamiento

mediciones = []
for nombre, claves in CONSULTAS.items():
    for repeticion in range(REPETICIONES):
        desplazamiento = repeticion % len(motores)
        orden = motores[desplazamiento:] + motores[:desplazamiento]
        for motor in orden:
            inicio = perf_counter()
            resultado = ejecutar(motor, nombre)
            segundos = perf_counter() - inicio
            pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(resultado, claves))
            assert int(resultado['reportes'].sum()) == TOTAL_ESPERADO
            mediciones.append({'consulta': nombre, 'motor': motor,
                'repeticion': repeticion + 1, 'segundos': segundos,
                'filas_entrada': TOTAL_ESPERADO, 'grupos': len(resultado), 'equivalente': True})
            print(nombre, repeticion + 1, motor, round(segundos, 4), 's', flush=True)
            del resultado


## L10. Leer las tablas y guardar la evidencia

**Haz esto:** ejecuta la celda y revisa estas salidas:

1. Total por corte: debe coincidir con las filas descargadas de cada semestre.
2. Diez entidades con más reportes por corte: interpretar como volumen, no como
   tasa de riesgo; no tenemos aquí el denominador de afiliados o exposición.
3. Mediana, mínimo y máximo del tiempo de cada motor y consulta.
4. Factores CPU/GPU: mediana CPU dividida por mediana GPU. Si vale 2, GPU tardó
   la mitad; si vale 0,5, GPU tardó el doble que ese motor CPU.

Los CSV y el control JSON quedan en `kit/salidas/pandas/P04_update1/`.
Los tiempos de conversión están en los controles Parquet, separados de las
consultas. No confundas reducción de tamaño del archivo con aceleración.


In [ ]:
for nombre, resultado in referencias.items():
    resultado.to_csv(OUT / f'{nombre}_verificado.csv', index=False)
por_corte = referencias['territorio_mes'].groupby('corte_archivo')['reportes'].sum()
assert por_corte.to_dict() == {f['periodo']: f['filas'] for f in FUENTES}
display(por_corte)
top_entidades = (referencias['territorio_mes_entidad']
    .groupby(['corte_archivo', 'ent_nombre'], dropna=False, as_index=False)['reportes'].sum()
    .sort_values(['corte_archivo', 'reportes', 'ent_nombre'], ascending=[True, False, True])
    .groupby('corte_archivo', sort=False).head(10))
display(top_entidades)
top_entidades.to_csv(OUT / 'top10_entidades_por_corte.csv', index=False)

tiempos = pd.DataFrame(mediciones)
assert len(tiempos) == len(CONSULTAS) * len(motores) * REPETICIONES
resumen = tiempos.groupby(['consulta', 'motor'], as_index=False).agg(
    mediana_s=('segundos', 'median'), min_s=('segundos', 'min'), max_s=('segundos', 'max'))
comparacion = resumen.pivot(index='consulta', columns='motor', values='mediana_s')
comparacion['pandas_sobre_gpu'] = comparacion['pandas_cpu'] / comparacion['polars_gpu']
comparacion['polars_cpu_sobre_gpu'] = comparacion['polars_cpu'] / comparacion['polars_gpu']
display(resumen)
display(comparacion)

versiones = {}
for paquete in ['pandas', 'polars', 'pyarrow', 'cudf-polars-cu12', 'cudf-polars-cu13']:
    try:
        versiones[paquete] = metadata.version(paquete)
    except metadata.PackageNotFoundError:
        versiones[paquete] = None
control_lab = {'periodos': PERIODOS, 'reportes': TOTAL_ESPERADO,
    'descargas': descargas, 'conversiones': conversiones,
    'gpu_ejecutada_y_equivalente': gpu_verificada,
    'dispositivo_cuda': DISPOSITIVO_CUDA, 'entorno_gpu_verificado_previamente': True,
    'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'versiones': versiones, 'python': sys.version, 'plataforma': platform.platform(),
    'cpu_logicas': os.cpu_count(), 'hilos_polars': pl.thread_pool_size(),
    'bloque_pandas': BLOQUE, 'repeticiones': REPETICIONES,
    'frontera': 'Parquet → agregación → resultado pandas en CPU',
    'descarga_y_conversion_incluidas_en_benchmark': False,
    'ram_pico_bytes': None, 'vram_pico_bytes': None}
tiempos.to_csv(OUT / 'benchmark_pandas_polars_gpu.csv', index=False)
pd.DataFrame(primeras).to_csv(OUT / 'primeras_ejecuciones.csv', index=False)
resumen.to_csv(OUT / 'resumen_benchmark.csv', index=False)
comparacion.to_csv(OUT / 'comparacion_medianas_y_factores.csv')
(OUT / 'control_laboratorio.json').write_text(json.dumps(control_lab, ensure_ascii=False, indent=2))
print('Evidencias:', OUT)


## L11. Explicar lo que pasó

Entrega las tablas y responde con tus resultados:

1. ¿Se descargaron y convirtieron los tres periodos? Mostrar los conteos.
2. ¿Qué hizo pandas con los bloques y qué hizo Polars con el plan lazy?
3. ¿Coincidieron los resultados de ambos ejercicios en los tres motores?
4. ¿Cuántos grupos agregó la consulta que incluye entidad?
5. ¿Qué motor ganó en cada consulta y cuál fue el factor CPU/GPU?
6. ¿Cómo explicas el resultado considerando lectura, transferencias y trabajo de agrupación?
7. ¿Qué archivos se reutilizan al ejecutar el laboratorio por segunda vez?

Adjunta tus CSV y `control_laboratorio.json` al informe de P04. No necesitas
repetir otros notebooks para completar este laboratorio. Si una ejecución GPU
falló, indica el error y no inventes tiempos ni factores.
